# 🎬 Movie Recommendation System

A content-based movie recommendation system using movie metadata, text preprocessing, CountVectorizer, and cosine similarity.

**Main pipeline:** Data → Cleaning → Feature Engineering → Text Preprocessing → Vectorization → Cosine Similarity → Recommendations


## 1. Import Libraries

In [1]:
import ast
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem.porter import PorterStemmer


## 2. Load the Datasets

In [2]:
credits_df = pd.read_csv("credits.csv")
movies_df = pd.read_csv("movies.csv")

print("Credits shape:", credits_df.shape)
print("Movies shape:", movies_df.shape)


Credits shape: (4803, 4)
Movies shape: (4803, 20)


## 3. Inspect the Data

In [3]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

display(movies_df.head())
display(credits_df.head())


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",10-12-2009,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",19-05-2007,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",26-10-2015,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",16-07-2012,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",07-03-2012,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [4]:
movies_df.info()
credits_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 
 6   original_title        4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   object 
 10  production_countries  4803 non-null   object 
 11  release_date          4802 non-null   object 
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   object 
 15  status               

## 4. Merge the Movie and Credits Data

In [5]:
movies_df = movies_df.merge(credits_df, on="title")

print("Merged shape:", movies_df.shape)
display(movies_df.head())


Merged shape: (4808, 23)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",10-12-2009,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",19-05-2007,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,285,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",26-10-2015,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,206647,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",16-07-2012,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106,49026,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",07-03-2012,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124,49529,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


## 5. Select Relevant Features

In [6]:
movies_df = movies_df[
    ["movie_id", "title", "overview", "genres", "keywords", "cast", "crew"]
].copy()

movies_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4808 entries, 0 to 4807
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  4808 non-null   int64 
 1   title     4808 non-null   object
 2   overview  4805 non-null   object
 3   genres    4808 non-null   object
 4   keywords  4808 non-null   object
 5   cast      4808 non-null   object
 6   crew      4808 non-null   object
dtypes: int64(1), object(6)
memory usage: 263.1+ KB


## 6. Handle Missing Values and Duplicates

In [7]:
print("Missing values before cleaning:")
display(movies_df.isnull().sum())

movies_df = movies_df.dropna().drop_duplicates().reset_index(drop=True)

print("\nShape after cleaning:", movies_df.shape)
print("\nMissing values after cleaning:")
display(movies_df.isnull().sum())


Missing values before cleaning:


movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64


Shape after cleaning: (4805, 7)

Missing values after cleaning:


movie_id    0
title       0
overview    0
genres      0
keywords    0
cast        0
crew        0
dtype: int64

## 7. Convert JSON-like Columns into Python Lists

In [8]:
def parse_list_column(value):
    """Convert a JSON-like string of dictionaries into a list of names."""
    try:
        if isinstance(value, list):
            return value
        data = ast.literal_eval(value)
        return [item["name"] for item in data if isinstance(item, dict) and "name" in item]
    except (ValueError, SyntaxError, TypeError):
        return []

def parse_cast(value, limit=3):
    """Extract the first few cast members."""
    try:
        if isinstance(value, list):
            data = value
        else:
            data = ast.literal_eval(value)
        return [
            item["name"]
            for item in data[:limit]
            if isinstance(item, dict) and "name" in item
        ]
    except (ValueError, SyntaxError, TypeError):
        return []


## 8. Process Genres, Keywords and Cast

In [9]:
movies_df["genres"] = movies_df["genres"].apply(parse_list_column)
movies_df["keywords"] = movies_df["keywords"].apply(parse_list_column)
movies_df["cast"] = movies_df["cast"].apply(parse_cast)

display(movies_df[["title", "genres", "keywords", "cast"]].head())


,title,genres,keywords,cast
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]"
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]"
2,Spectre,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux]"
3,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman]"
4,John Carter,"[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton]"


## 9. Extract the Director

In [10]:
def fetch_director(value):
    try:
        if isinstance(value, list):
            data = value
        else:
            data = ast.literal_eval(value)

        for item in data:
            if isinstance(item, dict) and item.get("job") == "Director":
                return item.get("name", "")
        return ""
    except (ValueError, SyntaxError, TypeError):
        return ""

movies_df["crew"] = movies_df["crew"].apply(fetch_director)

display(movies_df[["title", "crew"]].head())


,title,crew
0,Avatar,James Cameron
1,Pirates of the Caribbean: At World's End,Gore Verbinski
2,Spectre,Sam Mendes
3,The Dark Knight Rises,Christopher Nolan
4,John Carter,Andrew Stanton


## 10. Process the Movie Overview

In [11]:
movies_df["overview"] = (
    movies_df["overview"]
    .fillna("")
    .astype(str)
    .apply(lambda text: text.split())
)

display(movies_df[["title", "overview"]].head())


,title,overview
0,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."


## 11. Create the Combined Tags Feature

In [12]:
movies_df["tags"] = (
    movies_df["overview"]
    + movies_df["genres"]
    + movies_df["keywords"]
    + movies_df["cast"]
    + movies_df["crew"].apply(lambda x: [x] if x else [])
)

new_df = movies_df[["movie_id", "title", "tags"]].copy()

new_df["tags"] = new_df["tags"].apply(lambda words: " ".join(words).lower())

display(new_df.head())


,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


## 12. Text Stemming

In [13]:
ps = PorterStemmer()

def stem(text):
    return " ".join(ps.stem(word) for word in text.split())

new_df["tags"] = new_df["tags"].apply(stem)

display(new_df.head())


,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a parapleg marin is dispa..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believ to be dead, ha c..."
2,206647,Spectre,a cryptic messag from bond’ past send him on a...
3,49026,The Dark Knight Rises,follow the death of district attorney harvey d...
4,49529,John Carter,"john carter is a war-weary, former militari ca..."


## 13. Convert Text into Numerical Vectors

In [14]:
cv = CountVectorizer(
    max_features=5000,
    stop_words="english"
)

vectors = cv.fit_transform(new_df["tags"]).toarray()

print("Vector shape:", vectors.shape)
print("Number of features:", len(cv.get_feature_names_out()))


Vector shape: (4805, 5000)
Number of features: 5000


## 14. Calculate Cosine Similarity

In [15]:
similarity = cosine_similarity(vectors)

print("Similarity matrix shape:", similarity.shape)


Similarity matrix shape: (4805, 4805)


## 15. Build the Recommendation Function

In [16]:
def recommend(movie, n=5):
    # Match the movie title case-insensitively
    matches = new_df.index[
        new_df["title"].str.lower() == movie.lower()
    ]

    if len(matches) == 0:
        print(f"Movie '{movie}' was not found.")
        return []

    movie_index = matches[0]
    distances = similarity[movie_index]

    # Sort by similarity score and skip the input movie itself
    movie_list = sorted(
        list(enumerate(distances)),
        key=lambda x: x[1],
        reverse=True
    )[1:n+1]

    recommendations = []

    for index, score in movie_list:
        recommendations.append({
            "title": new_df.iloc[index]["title"],
            "similarity_score": round(float(score), 4)
        })

    return recommendations


## 16. Test the Recommendation System

In [17]:
recommend("Avatar", n=5)


[{'title': 'Aliens', 'similarity_score': 0.4936},
 {'title': 'Silent Running', 'similarity_score': 0.4113},
 {'title': 'Moonraker', 'similarity_score': 0.3881},
 {'title': 'Alien', 'similarity_score': 0.388},
 {'title': 'Mission to Mars', 'similarity_score': 0.3804}]

In [18]:
recommend("Thor", n=5)


[{'title': 'Thor: The Dark World', 'similarity_score': 0.5495},
 {'title': 'Avengers: Age of Ultron', 'similarity_score': 0.3839},
 {'title': 'Iron Man 2', 'similarity_score': 0.3685},
 {'title': 'Ant-Man', 'similarity_score': 0.3665},
 {'title': 'The Incredible Hulk', 'similarity_score': 0.3335}]

## 🔍 How the System Works

1. Load movie and credits datasets.
2. Merge the datasets using movie title.
3. Select relevant metadata.
4. Clean missing and duplicate records.
5. Extract genres, keywords, cast, and director.
6. Combine metadata into a single `tags` feature.
7. Apply text preprocessing and stemming.
8. Convert text into vectors using `CountVectorizer`.
9. Calculate pairwise similarity using cosine similarity.
10. Recommend the most similar movies.
